# 03 — From-scratch DPO: HH-RLHF and UltraFeedback (Task 4)

Accelerator: **GPU T4 x2** (policy on cuda:0, frozen reference on cuda:1), Internet **on**.

**Input:** attach the saved version of the SFT notebook (02) via *Add Input → Your Work*. This notebook
finds `outputs/sft/checkpoints/best` inside it and reuses its `data/processed` subsets.

Each run: 15,000 pairs, beta 0.1, fixed LR 1e-6, effective batch 32 (1 pair x 32 accumulation), 1 epoch
(469 optimizer steps), validation every 50 steps, best checkpoint by validation reward accuracy.
Expected ~2–2.5 h per run. Set `TRACKS` below to run one or both. When done: **Save Version**.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Locate the SFT checkpoint and data from the attached SFT notebook output

In [ ]:
import glob, os, shutil

TRACKS = ["hh", "uf"]   # e.g. ["hh"] to run only HH-RLHF in this session

matches = glob.glob("/kaggle/input/**/outputs/sft/checkpoints/best/model.safetensors", recursive=True)
assert matches, "SFT checkpoint not found: attach the SFT notebook's saved version as an input"
sft_root = os.path.dirname(os.path.dirname(os.path.dirname(matches[0])))   # .../outputs/sft
input_repo = os.path.dirname(os.path.dirname(sft_root))
print("SFT checkpoint:", os.path.dirname(matches[0]))

os.makedirs("outputs/sft", exist_ok=True)
if not os.path.exists("outputs/sft/checkpoints"):
    os.symlink(os.path.join(sft_root, "checkpoints"), "outputs/sft/checkpoints")
!ls -la outputs/sft/checkpoints/best/

for track in TRACKS:
    source = os.path.join(input_repo, "data", "processed", track)
    target = os.path.join("data", "processed", track)
    if os.path.isdir(source) and not os.path.isdir(target):
        shutil.copytree(source, target)
missing = [t for t in TRACKS if not os.path.exists(f"data/processed/{t}/train.jsonl")]
print("copied processed data for", [t for t in TRACKS if t not in missing], "| missing:", missing)

In [ ]:
# Only needed if the attached input had no data/processed for a track (rebuilds the identical seeded subsets).
if missing:
    !python scripts/prepare_data.py --tasks {" ".join(missing)} 2>&1 | grep -v Warning
!wc -l data/processed/*/train.jsonl data/processed/*/val.jsonl

## 2. DPO training
Logs every 5 optimizer steps (loss, reward accuracy, margin, grad norm, peak memory) and validates every 50.
Step 0 must show loss 0.6931, accuracy 0.000, margin 0.0000 (policy identical to reference).
If cuda:0 runs out of memory, append `gradient_checkpointing=true` to the command.

In [ ]:
!mkdir -p outputs/logs
if "hh" in TRACKS:
    !PYTHONUNBUFFERED=1 python scripts/train_dpo.py --config configs/dpo_hh.yaml 2>&1 | grep -v -i warn | tee outputs/logs/train_dpo_hh.log

In [ ]:
if "uf" in TRACKS:
    !PYTHONUNBUFFERED=1 python scripts/train_dpo.py --config configs/dpo_uf.yaml 2>&1 | grep -v -i warn | tee outputs/logs/train_dpo_uf.log

In [ ]:
import json
import matplotlib.pyplot as plt

for track in TRACKS:
    path = f"outputs/dpo_{track}/metrics.jsonl"
    if not os.path.exists(path):
        continue
    rows = [json.loads(line) for line in open(path)]
    train = [r for r in rows if r["type"] == "train"]
    evals = [r for r in rows if r["type"] == "eval"]
    summary = json.load(open(f"outputs/dpo_{track}/summary.json"))
    print(track, "best step", summary["best_step"], "best val acc", round(summary["best_val_reward_accuracy"], 3),
          "| final", {k: round(v, 4) for k, v in summary["final_val_metrics"].items()})

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for ax, key, label in zip(axes, ["loss", "reward_accuracy", "reward_margin"], ["DPO loss", "reward accuracy", "reward margin"]):
        ax.plot([r["step"] for r in train], [r[f"train_{key}"] for r in train], alpha=0.5, label="train (per step)")
        ax.plot([r["step"] for r in evals], [r[f"val_{key}"] for r in evals], "o-", label="validation")
        ax.set_xlabel("optimizer step")
        ax.set_title(f"DPO {track.upper()}: {label}")
        ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Sanity check: 3 prompts from each DPO best checkpoint

In [ ]:
if "hh" in TRACKS:
    !python scripts/generate.py --model_path outputs/dpo_hh/checkpoints/best --generator qwen2.5-0.5b-dpo-hh --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sanity_dpo_hh.json --limit 3 --print --overwrite 2>&1 | grep -v -i warn
if "uf" in TRACKS:
    !python scripts/generate.py --model_path outputs/dpo_uf/checkpoints/best --generator qwen2.5-0.5b-dpo-uf --eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_dpo_uf.json --limit 3 --print --overwrite 2>&1 | grep -v -i warn

## 4. Archive the small artifacts
Checkpoints stay in the saved notebook output; the zip holds metrics, configs, summaries, and logs.

In [ ]:
!rm -f outputs/generations/sanity_dpo_*.json
!zip -qr /kaggle/working/dpo_run_outputs.zip outputs/dpo_*/metrics.jsonl outputs/dpo_*/config.json outputs/dpo_*/summary.json outputs/dpo_*/checkpoints/*/training_state.json outputs/logs/train_dpo_*.log
!ls -lh /kaggle/working/*.zip && du -sh outputs/dpo_*/checkpoints/*